# Phase 2 — Notebook 04: Embedding Cache

**Goal:** Cache embeddings in Redis by content SHA-256 hash to avoid re-embedding identical content. Test cache hit/miss behaviour and measure performance gain.

**Packages used:** `redis`, `sentence-transformers`, `numpy`

> ⚠️ Requires: `uv add redis` and Redis container running (`docker-compose up -d redis`)

## 1. Setup

In [ ]:
import sys, time, hashlib, json
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from core.config import get_settings
settings = get_settings()

print(f"Redis URL  : {settings.redis_url}")
print(f"Cache TTL  : {settings.cache_ttl_seconds}s")

## 2. Redis Connection

In [ ]:
import redis

r = redis.from_url(settings.redis_url, decode_responses=False)
pong = r.ping()
print(f"Redis PING: {pong}")
print(f"Redis info: {r.info('server')['redis_version']}")

## 3. Cache Key Design

Key = `emb:{model_version}:{sha256(content)}`  
This ensures cached embeddings are invalidated automatically if the model changes.

In [ ]:
import numpy as np

def make_cache_key(text: str, model_version: str) -> str:
    content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return f"emb:{model_version}:{content_hash}"

def cache_set(r, key: str, embedding: list[float], ttl: int) -> None:
    # Serialize as JSON — compact and readable for inspection
    r.setex(key, ttl, json.dumps(embedding))

def cache_get(r, key: str) -> list[float] | None:
    raw = r.get(key)
    if raw is None:
        return None
    return json.loads(raw)

# Test key generation
sample = "Retrieval-augmented generation is a powerful technique."
key = make_cache_key(sample, settings.embedding_model_version)
print(f"Cache key: {key}")

## 4. Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.embedding_model)
print(f"Model loaded: {settings.embedding_model}")

## 5. Cache-Aware Embed Function

In [ ]:
def embed_with_cache(texts: list[str], model, r, model_version: str, ttl: int):
    results = []
    hits, misses = 0, 0

    for text in texts:
        key = make_cache_key(text, model_version)
        cached = cache_get(r, key)

        if cached is not None:
            results.append(cached)
            hits += 1
        else:
            embedding = model.encode(text, normalize_embeddings=True).tolist()
            cache_set(r, key, embedding, ttl)
            results.append(embedding)
            misses += 1

    return results, hits, misses

print("embed_with_cache defined")

## 6. Test Cache Miss (First Run)

In [ ]:
texts = [
    "Vector databases enable efficient similarity search.",
    "Large language models generate coherent text.",
    "Semantic chunking preserves meaning across chunk boundaries.",
    "pgvector extends PostgreSQL with vector similarity search.",
    "Retrieval-augmented generation grounds LLM answers in documents.",
]

# Clear any existing cache for clean test
for text in texts:
    r.delete(make_cache_key(text, settings.embedding_model_version))

t0 = time.time()
embeddings, hits, misses = embed_with_cache(
    texts, model, r, settings.embedding_model_version, settings.cache_ttl_seconds
)
elapsed_miss = time.time() - t0

print(f"First run (all misses):")
print(f"  Cache hits  : {hits}")
print(f"  Cache misses: {misses}")
print(f"  Time        : {elapsed_miss:.3f}s")

## 7. Test Cache Hit (Second Run)

In [ ]:
t0 = time.time()
embeddings_cached, hits, misses = embed_with_cache(
    texts, model, r, settings.embedding_model_version, settings.cache_ttl_seconds
)
elapsed_hit = time.time() - t0

print(f"Second run (all hits):")
print(f"  Cache hits  : {hits}")
print(f"  Cache misses: {misses}")
print(f"  Time        : {elapsed_hit:.4f}s")
print(f"\nSpeedup: {elapsed_miss / elapsed_hit:.1f}x faster on cache hit")

## 8. Verify Cached Embeddings are Identical

In [ ]:
all_match = all(
    np.allclose(embeddings[i], embeddings_cached[i], atol=1e-6)
    for i in range(len(texts))
)
print(f"Cached embeddings identical to original: {'✅ Yes' if all_match else '❌ No'}")

## 9. Model Version Isolation Test

Verify that a different model version produces a different cache key — preventing cross-model cache pollution.

In [ ]:
text = "Same text, different model."

key_v1 = make_cache_key(text, "bge-large-en-v1.5")
key_v2 = make_cache_key(text, "bge-base-en-v1.5")

print(f"Key (large): {key_v1}")
print(f"Key (base) : {key_v2}")
print(f"Keys differ: {'✅ Yes' if key_v1 != key_v2 else '❌ No'}")

## 10. Inspect Redis Key TTL

In [ ]:
key = make_cache_key(texts[0], settings.embedding_model_version)
ttl_remaining = r.ttl(key)
print(f"Key    : {key}")
print(f"TTL    : {ttl_remaining}s remaining (configured: {settings.cache_ttl_seconds}s)")
print(f"Size   : {r.strlen(key)} bytes")

## Summary

| Check | Result |
|---|---|
| Redis connection | ✅ |
| Cache miss → embed + store | ✅ |
| Cache hit → retrieve (no model call) | ✅ |
| Cached vectors identical | ✅ |
| Model version isolated in key | ✅ |
| TTL applied correctly | ✅ |

**Next:** `05_storage.ipynb` — insert chunks into Postgres and verify triggers/indexes.